### Practice: Large Language Models and Their Implications
<!-- ![img](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F4470ce74-e595-4750-92a5-5f21f040df6d_577x432.jpeg) -->
![img](https://i.imgur.com/QGYa2J8.jpeg)

In this notebook, you're gonna play with some of the largest language models on the Internet.

_Based on works of: Tim Dettmers, Ruslan Svirschevsky, Artem Chumachenko, Younes Belkada, Felix Marty, Yulian Gilyazev, Gosha Zolotov, Andrey Ishutin,  Elena Volf, Artemiy Vishnyakov, Svetlana Shirokovskih, Maria Ivanova.

### Part 1: prompt engineering (4 points total)

Your task is to prompt-engineer a **base** language model into solving a few tasks for you. A base model only *continues text*, it was never trained to help you, so you steer it with careful prompts and few-shot examples.

__Which model?__ run a base model locally, for example [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base).

__Deliverable:__ for each of the 4 problems, show your **prompt(s)**, the **code** you ran, the **model's output**, and a short **description** of your solution.

__How a base model behaves:__ it continues whatever you write. Begin a dialogue and it continues the dialogue;
supply a few worked examples and it imitates their format. Load the model in the two cells below, then use the
`generate()` helper in your solutions.

__It is fine to re-run and tweak.__ Base models are sensitive to wording: if the output is off, adjust the prompt
or add a few-shot example. If you need many retries for one task, your prompt probably needs rethinking.

#### Preparation


In [1]:
%pip install -q "transformers>=5.0" "accelerate>=1.0"
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm, trange

device = 'cuda'
major, _ = torch.cuda.get_device_capability()
dtype = torch.bfloat16 if major >= 8 else torch.float16   # T4 (compute 7.5) has no native bf16 -> use fp16
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(), "| dtype:", dtype)


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/home/bojkoar/ysda_nlp_course/.venv3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.20.2 | GPU: NVIDIA RTX A4000 | dtype: torch.bfloat16


In [2]:
torch.cuda.empty_cache()

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device).eval()


def generate(prompt, max_new_tokens=64, **kwargs) -> str:
    """Continue `prompt` with the base model; return only the newly generated text.
    Pass do_sample=True (+ temperature/top_p/top_k) for creative output, or nothing for greedy."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# smoke test: a base model just continues the text
print(generate("Once upon a time, deep in the forest,", max_new_tokens=40,
               do_sample=True, temperature=0.8, top_p=0.9))

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 4381.32it/s]


 there lived a giant wolf. The wolf had a big mouth and a big belly. He loved to eat sheep. He ate a lot of sheep, and his belly got bigger and bigger. One day


In [4]:
def print_generation(prompt: str, generated: str, color_code: int = 93):
    print(prompt, generated, sep=f"\x1b[{color_code};m", end="\x1b[0;m\n")

In [5]:
from difflib import ndiff

def highlight_diff(string: str, comp_with: str, color_code: int = 94) -> str:
    diff = ndiff(comp_with.split(), string.split())
    output_result = []
    for elem in diff:
        if elem[0] in "-?": continue
        if elem[0] == " ": output_result.append(elem[2:])
        else: output_result.append(f"\x1b[{color_code};m{elem[2:]}\x1b[0;m")
    return " ".join(output_result)

#### Task 1
__Task 1 (1 pt):__ arange a conversation between any two of the following:

- a celebrity or politician of your choice
- any fictional character (except Darth Vader)
- yourself

Compare two setups: a) you prompt with character names only b) you supply additional information (see example).

 __Example:__ Tony is talking to Darth Vader. Black text is written manually, blue text is generated.

![img](https://i.imgur.com/a1QhKF7.png)

In [ ]:
info = "Тёма Бойко студент лингвистики в Высшей школе экономики. Нюша это розовый смешарик, она любит красоваться и есть конфеты."
def make_prompt(provided_info: str): 
    return f"""
Тёма Бойко встречает Нюшу. {provided_info}

Тёма: Ничего себе, это что настоящая Нюша?!
Нюша:
""".strip()

for provided_info in ["", info]:
    prompt = make_prompt(provided_info)
    print_generation(
        prompt, 
        generate(prompt, max_new_tokens=100, do_sample=True, temperature=0.8, top_p=0.9)
    )
    print("\n")

Тёма Бойко встречает Нюшу. 

Тёма: Ничего себе, это что настоящая Нюша?!
Нюша: Да, это я.
Тёма: А ты как живешь?
Нюша: Хорошо, а ты?
Тёма: Ну, по твоим критериям, я живу как в твоём любимом сериале.
Нюша: Ну, хорошо. А ты, наверное, всегда хотел стать таким гением, как я?
Тёма: Ну, хотя бы я бы хотел быть немного счастливее.


Тёма Бойко встречает Нюшу. Тёма Бойко студент лингвистики в Высшей школе экономики. Нюша это розовый смешарик, она любит красоваться и есть конфеты.

Тёма: Ничего себе, это что настоящая Нюша?!
Нюша: Это я, Нюша, а ты кто?
Тёма: Ты как попала в этот учебный центр?
Нюша: Я живу в другом мире, но мы тут как-то договорились встретиться.
Тёма: Это что-то невероятное, я даже не знаю, что сказать.
Нюша: А ты что-нибудь хочешь попробовать? Может, у тебя есть какие-то вкусности?





#### Task 2
__Please choose task 2a or 2b (1pt)__ depending on your model (you can do both, but you will be awarded points for one of these two tasks).

__Task 2a:__ zero-shot translation. Take the first verse of [Edgar Allan Poe's "Raven"](https://www.poetryfoundation.org/poems/48860/the-raven) and __translate it into French.__ (You are free to use any other text of at least the same size)

Original text:

```
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
```

Verify your translation by converting french back into english using a public machine translation service.

__Task 2b:__ toxicity classification for [SetFit/toxic_conversations](https://huggingface.co/datasets/SetFit/toxic_conversations).

Make the model solve binary classification (toxic vs not toxic) in the few shot mode. For few-shot examples, use 2-3 toxic and 2-3 non-toxic non-toxic examples. Measure accuracy on at least 25 samples. You may need to try several different prompts before you find the one that works.

In [25]:
samples = [
    {
        "question": (
            "ребз, где в доке за подобное почитать? pytba вообще поддерживает такое?\n"
            "(спрашиваю в таком формате ибо не ебу даже как это называется)"
        ),
        "answer": (
            "https://pytba.readthedocs.io/ru/latest/sync_version/index.html#telebot.TeleBot.send_message\n\n"
            "Скорее всего это receiver_user_id, попробуй"
        ),
        "toxic": False
    },
    {
        "question": "Есть ли возможность боту сделать из группы - форум?\nне нашел ничего по поводу этого",
        "answer": "плохо искал https://core.telegram.org/method/channels.toggleForum",
        "toxic": True
    },
    {
        "question": "есть какой-то метод для получения списка участников группы? В документации не нашел такого(",
        "answer": "потому что нет такого, не нужно это ботам",
        "toxic": True
    },
    {
        "question": "А еще вопрос такой, можно считать что antiflood является батчем?",
        "answer": (
            "Не. Он скорее очередь. Он не отправляет всё пачкой, он следит, "
            "чтобы при получении флудвейта бот не долбился, а подождал и отправил снова"
        ),
        "toxic": False
    },
    {
        "question": "Приветствую\nБизнес бот через business_message_handler может обрабатывать сообщения о передаче нфт подарков?",
        "answer": "мошенники же как-то юзают\nочевидно надо подключить 3 пункт, чтобы воровать",
        "toxic": True
    },
    {
        "question": "Кстати на сайте про существование ApiTelegramException вообще не говориться",
        "answer": "иногда бывает полезно смотреть исходники, если они есть",
        "toxic": True
    },
    { #7
        "question": (
            "Всем ку!\nКто уже подключал оплату в боте, хелпаните как правильно это делается. "
            "К примеру мне надо получать оплату по СБП, видел ботов в который делается ссылка на оплату через qr.nspk"
        ),
        "answer": (
            "в @BotFather в разделе Payments можно подключить провайдера, если хочешь делать всю оплату внутри тг. "
            "если хочешь просто кидать ссылку или qr, то необязательно подключать провайдера через Payments "
            "(например, если его нет в списке).\nв обоих случаях нужно завести самозанятость или ИП.\n"
            "В случае подключения провайдера через тг, формировать чеки нужно через метод @bot.send_invoice(), "
            "а проверять и подтверждать оплату через обработчик @bot.pre_checkout_query_handler()"
        ),
        "toxic": False
    },
    { #8
        "question": "есть какой то способ отлова добавления бота в чат?",
        "answer": (
            "@bot.message_handler(content_types=['new_chat_members'])\n"
            "def new_user(message: Message):\n"
            "    if message.from_user.id == BOT_ID:\n"
            "        bot.send_message(message.chat.id, 'Я родился')"
        ),
        "toxic": False
    },
    { #9
        "question": "добрый день. такой вопрос, сколько максимально топиков можно создать в телеграмм чате?",
        "answer": "больше 1",
        "toxic": True
    },
    { #10
        "question": "парни вопрос первый раз пытаюсь написать бота, вот написал я весь код для бота и куда мне его пихать?",
        "answer": "Желательно на компьютер",
        "toxic": True
    },
    { #11
        "question": (
            "А можете пожалуйста подсказать с помощью какого метода бизнесс аккаунт может "
            "отправлять подарки пользователю и получать его от них ?"
        ),
        "answer": "https://pytba.readthedocs.io/ru/latest/sync_version/index.html#telebot.TeleBot.transfer_gift",
        "toxic": False
    },
    { #12
        "question": (
            "Раньше все как часы работало, буквально час назад началась такая ошибка при передачи бизнес ботом NFT подарка. "
            "Что может быть не так? Chat ID точно верный\n"
            "A request to the Telegram API was unsuccessful. Error code: 400. "
            "Description: Bad Request: unknown chat identifier specified"
        ),
        "answer": ">Chat ID точно верный\nВыходит, что не точно",
        "toxic": True
    },
    { #13
        "question": "помогите пожалуйста  🙏\nвсе же правильно?\nне отправляет соо в ответ на «/start»",
        "answer": "1. commands=\n2. убери /start",
        "toxic": False
    },
    { #14
        "question": (
            "всем привет! хотел спросить: как выдавать должности в чате с помощью pytelegrambotapi? "
            "ну чтобы вместо админ писало что то другое"
        ),
        "answer": "https://core.telegram.org/bots/api#setchatadministratorcustomtitle",
        "toxic": False
    },
    { #15
        "question": (
            "всем здравствуйте. мне поступил заказ на создание бота, используя фичу телеграма, как «Telegram Business». "
            "я ранее не слышал про неё, ну и следовательно ничего не делал с ним. хочу узнать, telebot вообще поддерживает это "
            "и если да, то можете скинуть какой-нибудь шаблон с этой штукой?"
        ),
        "answer": "на гитхабе разве нет примеров?",
        "toxic": True
    },
    { #16
        "question": "боты могут создавать премиум емодзи? Или только стикеры? Как называются методы?",
        "answer": (
            "Покупаешь ник, за сколько то там ТОН(вроде за тыщу) ставишь ник на бота "
            "и тогда для тебя открываются прелести премиум эмоджи"
        ),
        "toxic": False
    },
    { #17
        "question": "а есть мб на офф страницах телеги эта инфа?\nя уверен что мимо проходил и просто не понял)\nтуповат",
        "answer": "просто логически подумай\nпокопируй ссылки, посмотри на них",
        "toxic": True
    },
    { #18
        "question": (
            "ребята помогите со стейтами\nкак реализовать изменение предмета из базы с помощью стейтов\n"
            "есть отдельная линия стейтов для внесения, но не хочется прописывать отдельную линию стейтов "
            "для изменения предмета, хочется использовать уже готовые стейты"
        ),
        "answer": "Лень две строки дописать, но не лень всю логику менять?",
        "toxic": True
    },
    { #19
        "question": (
            "всем привет\nу меня есть чат с ботом, где есть закреп, в котором я хочу обновлять информацию.\n"
            "пытаюсь получить закреп:\nbot.get_chat(envs.ADMIN_USER_ID).pinned_message\nно эта переменная = None\n"
            "- чат точно верный, проверил в отладке (да и нет у бота других чатов сейчас)\n"
            "- закреп тоже точно есть, одно сообщение, никто его не снимал, сообщение не удалялось.\n\n"
            "можете что-то посоветовать?"
        ),
        "answer": (
            "Это ты де такой аттрибут нашел, ии помог?\n"
            "https://pytba.readthedocs.io/en/latest/types.html#telebot.types.ChatFullInfo"
        ),
        "toxic": True
    },
    { #20
        "question": "Как запустить асинхронного бота без блокировки основного потока?",
        "answer": (
            "Asyncio.gather например\nhttps://github.com/SeriouSad/telegram-tracker-agent/blob/main/main.py\n"
            "Там не telebot конечно, но принцип схожий"
        ),
        "toxic": False
    },
    { #21
        "question": "Как отличить Too many requests  от другой ApiTelegramException?",
        "answer": "глазами\nтам же есть всякие коды и описания",
        "toxic": True
    },
    { #22
        "question": (
            "Можно ли внутри хендлера перезапустить процесс обработки сообщения?\n"
            "Что-то вроде RestartHandling() по аналогии с ContinueHandling()."
        ),
        "answer": "готовых способов нет\nзачем бы это кому-то понадобилось?",
        "toxic": True
    },
    { #23
        "question": (
            "привет всем, меня заинтересовала фишка, когда бот управляет чатом(телеграм бот для бизнеса) "
            "я не очень понимаю, как это применить к коду\n"
            "например, у меня подключен бот, как можно сделать, когда мне пишет человек в личные сообщения "
            "бот бы отвечал ему? может есть документация или пример кода?"
        ),
        "answer": "1:1 как с обычным,  ток хендлер другой",
        "toxic": False
    },
    { #24
        "question": (
            "Всем привет. Не смог нарыть ответ: есть ли возможность получить текущий username (т.е. @…) пользователя по user_id?"
        ),
        "answer": "от апи нельзя, только сохранять самому в базу данных",
        "toxic": False
    },
    { #25
        "question": "Как удалить сообщение? Ловлю с помощью chat_member_handler",
        "answer": "эм",
        "toxic": True
    },
    { #26
        "question": (
            "почему я не могу использовать business методы кроме get_business_connection? "
            "выдает ошибки по типу AttributeError: 'TeleBot' object has no attribute 'get_business_account_star_balance'"
        ),
        "answer": "может версия либы старая",
        "toxic": False
    },
    { #27
        "question": "Подскажите пожалуйста,  после нажатия на колбек  кнопку,  как прекратить её мигание",
        "answer": "чтобы кнопка мигать перестала нужно ответить на коллбек либо клавиатуру поменять",
        "toxic": False
    },
    { #28
        "question": (
            "async def warn_1(call: ctx.CallbackQuery, state: ctx.StateContext):\n"
            "    text = '''Пожалуйста введите id пользователя:'''\n"
            "    state.set(ctx.MyStates.id_warn)\n"
            "    kb = await kb_state_canc()\n"
            "    ctx.bot.edit_message_text(text, call.id, call.message.message_id, reply_markup=kb)\n\n"
            "код вызывает ошибку:\n\n"
            "2025-04-07 18:08:25,470 (async_telebot.py:591 MainThread) ERROR - TeleBot: "
            "\"warn_1() missing 1 required positional argument: 'state'\n"
            "вот вызов(хандлер):\n"
            "ctx.bot.register_callback_query_handler(adm_warn,               lambda call: call.data == \"admin:warn\")\n"
            "а гпт выдал херню которая делает тоже самое только запускается через передачу стейта в функцию заного"
            "(def main(call:*, state:*): main_1(call, state) )"
        ),
        "answer": "надо питон учить наверное и английский\nпишет же, что не хватает аргумента при вызове функции",
        "toxic": True
    },
    { #29
        "question": (
            "При нажатии кнопки, я проверяю язык пользователя, но у некоторых вместо «ru en ar итд» возвращает None. "
            "Почему так ? И что с этим делать"
        ),
        "answer": (
            "можешь если None то ставить свой дефолтный язык какой-то, типа английский к примеру\n",
            "ты никак не можешь на это повлиять потому что апи берет язык от клиента юзера"
        ),
        "toxic": False
    },    
]


In [41]:
def sample_to_str(sample: dict[str, str | bool], hide_class: bool):
    return """
<question>{q}</question>
<answer>{a}</answer>
<is_toxic>{t}""".format(
    q=sample["question"], 
    a=sample["answer"], 
    t="" if hide_class else str(sample["toxic"]) + "</is_toxic>"
).lstrip()

EXAMPLES = "\n\n".join([sample_to_str(sample, hide_class=False) for sample in samples[:4]]) + "\n\n"
correct = 0
total = 0
errors = 0
for sample in samples[4:]:
    test_case = sample_to_str(sample, hide_class=True)
    prompt = EXAMPLES + test_case
    pred = generate(prompt, max_new_tokens=1)
    pred_val = True if pred=="True" else False if pred=="False" else None
    if total == 0:
        print_generation(prompt, pred)
    elif pred_val is None:
        print_generation(prompt, pred, color_code=91)
    total += 1
    if pred_val is None:
        errors = pred_val
    elif pred_val == sample["toxic"]:
        correct += 1

accuracy = correct / total

print(f"{accuracy=}")
print(f"{errors} error(s)")

<question>ребз, где в доке за подобное почитать? pytba вообще поддерживает такое?
(спрашиваю в таком формате ибо не ебу даже как это называется)</question>
<answer>https://pytba.readthedocs.io/ru/latest/sync_version/index.html#telebot.TeleBot.send_message

Скорее всего это receiver_user_id, попробуй</answer>
<is_toxic>False</is_toxic>

<question>Есть ли возможность боту сделать из группы - форум?
не нашел ничего по поводу этого</question>
<answer>плохо искал https://core.telegram.org/method/channels.toggleForum</answer>
<is_toxic>True</is_toxic>

<question>есть какой-то метод для получения списка участников группы? В документации не нашел такого(</question>
<answer>потому что нет такого, не нужно это ботам</answer>
<is_toxic>True</is_toxic>

<question>А еще вопрос такой, можно считать что antiflood является батчем?</question>
<answer>Не. Он скорее очередь. Он не отправляет всё пачкой, он следит, чтобы при получении флудвейта бот не долбился, а подождал и отправил снова</answer>
<is_tox

> 0,76! Нормально так для модели без обучения

#### Task 3
__Task 3 (1pt):__ create a prompt and few-shot examples tha make the model __change the gender pronouns__ of the main actor in a given sentence in any direction of your choice. E.g. the doctor took off _his_ mask <-> the doctor took of _her_ mask.


In [91]:
import re

EXAMPLES = """
<input>The doctor took off his mask</input gender=male>
<output>The doctor took off her mask</output gender=female>

<input>My friend called her mother and told her that she would come home late</input gender=female>
<output>My friend called his mother and told her that he would come home late</output gender=male>

<input>Our teacher hurt himself badly during a demonstration. He is in the hospital right now</input gender=male>
<output>Our teacher hurt herself badly during a demonstration. She is in the hospital right now</output gender=female>

""".lstrip()

def change_the_gender(input_string: str, output_generation: bool = False):
    prompt = EXAMPLES + f"<input>{input_string}</input gender="
    generation = ""
    while "</output" not in generation:
        generation += generate(prompt + generation, 10)
        if len(generation) > len(prompt):
            raise ValueError("Слишком длинная генерация:", generation)
    if output_generation:
        print_generation(prompt, generation)
        print()
    output = re.search(r"\<output\>(.+?)\<\/output", generation).group(1)
    return output

test_cases = [
    "I finally got to know my neighbour! She is very nice, she invited us to come visit her for her birthday on the weekend",
    "The speaker was weird. He made inappropriate jokes and his voice was strangely creaky.",
    "Il appert également au dossier que le prestataire était machiniste et qu'il a abandonné volontairement son emploi.",
    # ↑ It also appears in the record that the claimant was a machinist who voluntarily abandoned his employment.
    "Эта всезнайка меня уже достала, она постоянно придирается ко всем! Даже её подруга устала, её достали её занудства."
]

for i, test_case in enumerate(test_cases):
    result = change_the_gender(test_case, output_generation=(i == 0))
    print(highlight_diff(test_case, result, 91), "\n→", highlight_diff(result, test_case, 92))

<input>The doctor took off his mask</input gender=male>
<output>The doctor took off her mask</output gender=female>

<input>My friend called her mother and told her that she would come home late</input gender=female>
<output>My friend called his mother and told her that he would come home late</output gender=male>

<input>Our teacher hurt himself badly during a demonstration. He is in the hospital right now</input gender=male>
<output>Our teacher hurt herself badly during a demonstration. She is in the hospital right now</output gender=female>

<input>I finally got to know my neighbour! She is very nice, she invited us to come visit her for her birthday on the weekend</input gender=neutral>
<output>I finally got to know my neighbour! He is very nice, he invited us to come visit him for his birthday on the weekend</output gender=neutral>

<input>My friend

I finally got to know my neighbour! She is very nice, she invited us to come visit her for her birthday on the weekend 
→ I finally 

> Успешный успех! Работает даже с другими языками и не заменяет автоматом все местоимения, а выбирает по смыслу  
>
> При этом одни лишь примеры не приводили к успеху, моделька не понимала чё от неё требуют и просто повторяла последнее предложение. Помогло добавить пометку `gender=male/female`, которая подсказывает какие метаданные есть в текстовом поле и интересуют нас  
> Интересно, что почему-то оно пометку `gender=` продолжает через `neutral`. Но успешно выполнить задание это не мешает

#### Task 4
__Task 4 (1pt):__ write a prompt and supply examples such that the model would __convert imperial units to metric units__ (miles -> kilometers; mph -> kph). More specifically, the model should rewrite a given sentence and replace all imperial units with their metric equivalents. After it works with basic distances and speed, try to find complicated examples where it does *not* work.

Please note that 1 mile is not equal to 1 km :)

In [100]:
import re

EXAMPLES = """
<input>He couldn't have driven 18 miles in just 10 minutes!</input system=imperial>
<output>He couldn't have driven 29 kilometers in just 10 minutes!</output system=metric>

<input>My friend is 5'11" tall</input system=imperial>
<output>My friend is 180 cm tall</output system=metric>

<input>10 inches should be enough</input system=imperial>
<output>25 centimiters should be enough</output system=metric>

""".lstrip()

def change_to_metric(input_string: str, output_generation: bool = False):
    prompt = EXAMPLES + f"<input>{input_string}</input system=imperial>\n<output>"
    generation = ""
    while "</output" not in generation:
        generation += generate(prompt + generation, 10)
        if len(generation) > len(prompt):
            raise ValueError("Слишком длинная генерация:", generation)
    if output_generation:
        print_generation(prompt, generation)
        print()
    output = re.search(r"(.+?)\<\/output", generation).group(1)
    return output

test_cases = [
    "Stop when you see a bright convenience store to your right, turn left and go straight forward for 3 miles.", # 4.8 km
    "Do you know anyone shorter than 5'3\"??", # 160 cm
    "The girth of his forearm reaches 20 in, and his ring finger measures 5 inches.", # 50.8 cm, 12.7 cm
    "The peak of mount Aino is located at an elevation of 10,466 ft, making it the third tallest peak in Japan.", # 3,190 m
    "Its peak is located at an elevation of 10,466 ft.", # 3,190 m
    "Its peak is located at an elevation of 7,841 ft.", # 2,390 m
    "Диагональ 15.6 дюймов, матрица IPS — частота 1.2 ГГц." # 39.6 см
]

for i, test_case in enumerate(test_cases):
    result = change_to_metric(test_case, output_generation=(i == 0))
    print(highlight_diff(test_case, result, 91), "\n→", highlight_diff(result, test_case, 92))

<input>He couldn't have driven 18 miles in just 10 minutes!</input system=imperial>
<output>He couldn't have driven 29 kilometers in just 10 minutes!</output system=metric>

<input>My friend is 5'11" tall</input system=imperial>
<output>My friend is 180 cm tall</output system=metric>

<input>10 inches should be enough</input system=imperial>
<output>25 centimiters should be enough</output system=metric>

<input>Stop when you see a bright convenience store to your right, turn left and go straight forward for 3 miles.</input system=imperial>
<output>Stop when you see a bright convenience store to your right, turn left and go straight forward for 4.8 kilometers.</output system=metric>



Stop when you see a bright convenience store to your right, turn left and go straight forward for 3 miles. 
→ Stop when you see a bright convenience store to your right, turn left and go straight forward for 4.8 kilometers.
Do you know anyone shorter than 5'3"?? 
→ Do you know anyone shorter than 160 cm??

> Хорош даже с большими числами взятыми из головы. Либо модель просто запомнила кучу примеров и таблицы переводов, которых в инете тоже полно, либо всё же она умеет в математику

### Part 2: local inference & text-generation internals

In Part 1 we used `model.generate` as a black box. Now let's look **under the hood**: how a language model turns logits into text and implement a decoding strategy by hand.

We already loaded [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base) above: a dense 4B model that fits a free Colab T4 in fp16 (~8 GB).

#### Text generation

**Comparison of strategies for language model text generation:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br> **Cons:** Computationally more complex and requires a good loss function. |

Documentation references:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

#### Generation with HuggingFace

In [6]:
prompt = 'The first discovered martian lifeform looks like'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=True, temperature=0.8)
# greedy inference:                                        do_sample=False)
# beam search for highest probability:                     num_beams=4)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

Input batch (encoded): {'input_ids': tensor([[  785,  1156, 11105, 48195,  1103,  2272,   627,  5868,  1075]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

Output: The first discovered martian lifeform looks like a tiny blob of goo.
That’s the shape it has in the images from NASA’s Curiosity rover, which was the first to visit Mars in 2012. But that’s just how it looks. It’s really something altogether different. And it’s something scientists have been waiting for.
“This is a


#### Low-level code for text generation

In [7]:
prompt = "Moscow is the capital of"
# prompt = "Skippy, a young android, likes to dream about electric"

print(prompt, '\n')

voc = tokenizer.get_vocab()
voc_rev = {v:k for k, v in voc.items()}  # reverse vocab for decode

for i in range(10):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probs.flatten(), num_samples=1)

    next_token = tokenizer.decode(next_token_id)
    prompt += next_token

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    top_tokens = sorted_indices[:5]
    print(f"Step #{i} candidates:")
    for t, p in zip (top_tokens, sorted_probs):
        t = voc_rev[t.item()]
        print(f"{t:<10}: {p:.4f} ")

    print(f'\nChosen token: {next_token}', end='\n\n', flush=True)

Moscow is the capital of 

Step #0 candidates:
ĠRussia   : 0.4648 
Ġthe      : 0.1514 
Ġwhich    : 0.1514 
Ġwhat     : 0.0491 
Ġ______   : 0.0232 

Chosen token:  which

Step #1 candidates:
Ġcountry  : 0.6445 
ĠEuropean : 0.1631 
ĠRussian  : 0.0771 
Ġnation   : 0.0172 
ĠAsian    : 0.0118 

Chosen token:  country

Step #2 candidates:
?         : 0.2715 
?Ċ        : 0.2109 
?ĊĊ       : 0.1865 
,         : 0.1128 
???       : 0.1128 

Chosen token: ?



Step #3 candidates:
Mos       : 0.6367 
Russia    : 0.1426 
The       : 0.0977 
What      : 0.0219 
1         : 0.0080 

Chosen token: Mos

Step #4 candidates:
cow       : 1.0000 
k         : 0.0000 
quito     : 0.0000 
cou       : 0.0000 
ow        : 0.0000 

Chosen token: cow

Step #5 candidates:
Ġis       : 0.9961 
,         : 0.0019 
Ġserves   : 0.0008 
Ġwas      : 0.0002 
Ġ(        : 0.0001 

Chosen token:  is

Step #6 candidates:
Ġthe      : 1.0000 
Ġa        : 0.0004 
Ġindeed   : 0.0003 
Ġnot      : 0.0002 
Ġlocated  : 0.0001 

Chos

#### Task 5
**Task 5: write code for nucleus sampling generation (2 points)**:

Use the `nucleus_sampling()` template below. Look at the detailed generation code above for inspiration. __Please do not use model.generate__.

**Bonus task: write code for beam search (3 bonus points)**

In [35]:
from typing import Tuple, List

VOCAB = tokenizer.get_vocab()
# REVERSE_VOCAB = {v:k for k, v in voc.items()}  # reverse vocab for decode

def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> Tuple[str, List[str]]:
    """generates the next token from the nucleus of tokens with cumulative probability up to param:prob"""
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    cumulative_prob = torch.cumsum(sorted_probs, dim=0)
    last_ind = (cumulative_prob > prob).to(torch.int).argmax()
    top_token_probs = sorted_probs[:last_ind+1]
    # print(f"Step #{i} candidates:")
    # for t, p in zip (top_tokens, sorted_probs):
    #     t = voc_rev[t.item()]
    #     print(f"{t:<10}: {p:.4f} ")

    sampled_token_id = sorted_indices[torch.multinomial(top_token_probs, num_samples=1)]
    sampled_token = tokenizer.decode(sampled_token_id)

    possible_tokens = [tokenizer.decode(i) for i in sorted_indices[:last_ind+1]]

    # sampled_token should be a string token that was generated
    # possible_tokens should be a list of all tokens that have non-zero probability
    return sampled_token, possible_tokens

In [38]:
# Tests for nucleus sampling.
prompt = "Elbrus is the highest"

sampled, nucleus = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
print("sampled:", repr(sampled), "| nucleus:", nucleus, "| nucleus size:", len(nucleus))

# 1) the sampled token must come from inside the nucleus
assert sampled in nucleus
assert len(nucleus) >= 1

# 2) the nucleus depends only on `prob`, not on the random draw -> it is deterministic across calls
_, nucleus_again = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
assert sorted(nucleus) == sorted(nucleus_again), "the nucleus itself should not be random"

# 3) a larger probability threshold never yields a smaller nucleus
_, small = nucleus_sampling(model, tokenizer, prompt, prob=0.1)
_, large = nucleus_sampling(model, tokenizer, prompt, prob=0.95)
assert len(large) >= len(small), "higher p should include at least as many tokens"

# 4) a tiny threshold collapses to the single most likely token
_, top1 = nucleus_sampling(model, tokenizer, prompt, prob=1e-6)
assert len(top1) == 1, "with p -> 0 the nucleus is just the top-1 token"

print("All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)")

sampled: ' mountain' | nucleus: [' mountain', ' peak', ' point'] | nucleus size: 3
All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)


### Part 3: Chain-of-thought prompting (4 points total)

![img](https://github.com/kojima-takeshi188/zero_shot_cot/raw/main/img/image_stepbystep.png)

---



In [11]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"
!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
data = list(map(json.loads, open("aqua.json")))

--2026-11-05 22:44:56--  https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 130192 (127K) [text/plain]
Saving to: ‘aqua.json’

aqua.json           100%[===================>] 127.14K  --.-KB/s    in 0.04s   

2026-11-05 22:44:57 (3.29 MB/s) - ‘aqua.json’ saved [130192/130192]



In [12]:
print("Example:")
data[150]

Example:


{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?',
 'options': ['A)1 minute',
  'B)2 minutes',
  'C)3 minutes',
  'D)4 minutes',
  'E)5 minutes'],
 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B",
 'correct': 'B'}

### Naive solution

Here, we prompt the model to choose an answer to the example above (`data[150]`) out of the options given above. We're using a format that mimics grade school solution textbook.

Please note that there are minor formatting changes in options: an extra space and an opening bracket. Those may or may not be important :)

In [13]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

In [14]:
# solving an equation directly
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

[Prompt:]
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
[Generated:]  D
Answer: Janice bikes at 1 mile per 6 minutes, while Jennie bikes at 1 per 3 minutes. So together they cover 1/6 + 1/3 = 1/2 miles every three minutes. That means it will take them six minutes to cover one mile.<|endoftext|>


And here's how you can solve this with few-shot chain-of-thought prompting.

You need to chang 3 things
- use a new field called **Rationale**, that contains a step-by-step solution to the problem
- add several few-shot examples of previously solved problems **with rationales**
- change the final prompt so that the model has to generate rationale before answering

In [15]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [16]:
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

[Prompt:]
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If 

#### Task 6
__Task 6 (1 pt)__ write a function that automatically creates chain-of-thought prompts. Follow the instructions from the function docstring.

In [17]:
import re

QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

def make_prompt(*, main_question, fewshot_examples):
    """
    Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

    For each few-shot question, make sure to follow the following rules:
    1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
    2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
    3. Then, provide the answer as a single letter (A-E)
    4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

    Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
    The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
    After that, you should prompt the model to produce an explanation (rationale) for the answer.

    Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT

    question
    options
    rationale
    correct
    """

    def format_option(options: list[str]):
        return ' '.join([re.sub(r"^(\w)\)", r"(\1) ", o) for o in options])

    fewshot_formatted = [f"""
{QUESTION_PREFIX}{example['question']}
{OPTIONS_PREFIX}{format_option(example['options'])}
{CHAIN_OF_THOUGHT_PREFIX}{example['rationale']}
{ANSWER_PREFIX}{example['correct']}
""".strip() for example in fewshot_examples]
    fewshot_formatted.append(f"""
{QUESTION_PREFIX}{main_question['question']}
{OPTIONS_PREFIX}{format_option(main_question['options'])}
{CHAIN_OF_THOUGHT_PREFIX}""".strip())

    return FEWSHOT_SEPARATOR.join(fewshot_formatted)



generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))
assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, \
    (
        f"prompts don't match:\n"
        f"{highlight_diff(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, generated_fewshot_prompt, 91)}\n"
        f"{highlight_diff(generated_fewshot_prompt, EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, 92)}"
    )
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

# Hint: if two prompts do not match, you may find it usefull to use https://www.diffchecker.com or similar to find the difference

Well done!


#### Task 7
__Task 7 (1 points):__ Evaluate your prompt.

Please run the model on the entire dataset and measure it's accuracy.
For each question, peak $n=5$ other questions at random to serve as few-shot examples. Make sure not to accidentally sample the main_question among few-shot examples. For scientific evaluation, it is also a good practice to split the data into two parts: one for eval, and another for few-shot examples. However, doing so is optional in this homework.

The tricky part is when to stop generating: if you don't control for this, your model can accidentally generate a whole new question - and promptyly answer it :) To make sure you get the correct answer, stop generating tokens when the model is done explaining it's solution. To circumvent this, you need to __stop generating as soon as the model generates Final Answer: [A-E]__
To do so, you can either generate manually (see low-level generation above) or use [transformers stopping criteria](https://discuss.huggingface.co/t/implimentation-of-stopping-criteria-list/20040/2), whichever you prefer.

If you do everything right, the model should be much better than random. However, please __do not expect miracles__: this is far from the best models, and it will perform much worse than an average human.

In [18]:
NUM_SAMPLES = 0    # use this to count how many samples you evaluated
NUM_RESPONDED = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
NUM_CORRECT = 0    # how many times did the model's chosen answer (letter) match the correct answer

In [19]:
stopwords = torch.tensor(tokenizer.encode(FEWSHOT_SEPARATOR))
stopwords

tensor([1406])

In [70]:
from transformers import StoppingCriteria, StoppingCriteriaList

class StoppingCriteriaSub(StoppingCriteria):

    def __init__(self, stops = [], encounters=1):
        super().__init__()
        self.stops = stops
        self.ENCOUNTERS = encounters

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor):
        stop_count = 0
        for stop in self.stops:
            stop_count += (input_ids[0] == stop).sum().item()

        if stop_count >= self.ENCOUNTERS:
            return True
        return False

def stopping_criteria(n: int):
    return StoppingCriteriaList([StoppingCriteriaSub(stops=stopwords, encounters=n+1)])

In [71]:
result = generate(
    make_prompt(main_question=data[33], fewshot_examples=data[:5]), 
    max_new_tokens=200, 
    do_sample=True,
    top_p=0.25,
    temperature=0.5,
    stopping_criteria=stopping_criteria(5)
)
print_generation(data[5]["question"] + "\n" + ", ".join(data[5]["options"]) + "\n", result.strip())
match = re.search(r"Correct Answer: ([A-E])", result)
final_answer = match.group(1) if match else None
print(f"{final_answer=}")
print(f"last token: [{tokenizer.tokenize(result)[-1]}]")

20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
A)40, B)200, C)380, D)400, E)3200
The different ways in which a person can travel from M to Q
= (No. of ways from M to N) x (No. of ways from N to O) x (No. of ways from O to P) x (No. of ways from P to Q)
= 3 x 4 x 1 x 3
= 36
Answer : B
Correct Answer: B
final_answer='B'
last token: [ĊĊĊ]


In [72]:
from IPython.display import clear_output

def count_performance(data: list[dict[str, str | list[str]]], k_shots: int = 5):
    random.seed(239)
    weights = [1] * len(data)
    for i, question in enumerate(tqdm(data, "Inference")):
        weights[i] = 0
        fewshot_questions = random.choices(data, weights, k=k_shots)
        weights[i] = 1
        prompt = make_prompt(main_question=question, fewshot_examples=fewshot_questions)
        result = generate(
            prompt,
            max_new_tokens=1024, 
            do_sample=True,
            top_p=0.25,
            temperature=0.5,
            stopping_criteria=stopping_criteria(k_shots)
        )
        if result.endswith(FEWSHOT_SEPARATOR) and "Correct Answer:" not in result:
            result = result[:-len(FEWSHOT_SEPARATOR)] + "Correct Answer:"
            result += generate(prompt + result, max_new_tokens=1)
        match = re.search(r"Correct Answer: ([A-E])", result)
        final_answer = match.group(1) if match else None
        global NUM_SAMPLES
        NUM_SAMPLES += 1
        if final_answer is not None:
            global NUM_RESPONDED
            NUM_RESPONDED += 1
        else:
            clear_output(wait=True)
            print_generation(prompt.split("\n\n")[-1], result)
            print(f"Last token: [{tokenizer.tokenize(result)[-1].__repr__()}]")
        if final_answer == question["correct"]:
            global NUM_CORRECT
            NUM_CORRECT += 1

In [64]:
len(tokenizer.tokenize("""
Let the length and breadth of the rectangle be l and b respectively.
When folded along the length, the perimeter of each part = 2(l + b/2) = 48
=> 2l + b = 96
When folded along the breadth, the perimeter of each part = 2(b + l/2) = 66
=> 2b + l = 66
Solving the above equations, we get l = 30 and b = 36
Area of the rectangle = lb = 30 * 36 = 1080
The answer is E.
""".strip()))

132

In [65]:
count_performance(data)

# Optionally, consider inferencing multiple sentences in a batch for faster inference;
# If you choose to batch outputs, make sure the results are the same as with batch=1 (using greedy inference)

Inference:  13%|█▎        | 33/254 [04:19<19:55,  5.41s/it]


Question: When folded into two equal halves a rectangular sheet had a perimeter of 48cm for each part folded along one set of sides and the same is 66cm when folded along the other set of sides. Find the area of the sheet.
Answer Choices: (A) 1584 (B) 1120 (C) 792 (D) 1320 (E) 1200
Rationale: Let the length and breadth of the rectangle be l and b respectively.
When folded along the length, the perimeter of each part = 2(l + b/2) = 48
=> 2l + b = 96
When folded along the breadth, the perimeter of each part = 2(b + l/2) = 66
=> 2b + l = 66
Solving the above equations, we get l = 30 and b = 36
Area of the rectangle = lb = 30 * 36 = 1080
The answer is E.
Last token: ['.']


Inference:  13%|█▎        | 33/254 [05:02<33:48,  9.18s/it]


KeyboardInterrupt: 

In [ ]:
print("Responded %%:", NUM_RESPONDED / NUM_SAMPLES)
print("Accuracy (when responded):", NUM_CORRECT / NUM_RESPONDED)
print("Accuracy (overall):", NUM_CORRECT / NUM_SAMPLES)

if NUM_RESPONDED / NUM_SAMPLES < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.6805225653206651
Accuracy (when responded): 0.7591623036649214
Accuracy (overall): 0.5166270783847982
Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.
Make sure you generate enough tokens that the model can produce a correct answer.
When in doubt, take a look at the full model output. You can often spot errors there.


__Task 8 (2 points)__ Experiment time!

Your final quest is to use the testbench you've just written to answer one of the following questions:

### Option 1: How many shots do you need?

How does model accuracy change with the number of fewshot examples?

a. check if the model accuracy changes as you increase/decrease the number of "shots"

b. try to prompt-engineer a model into giving the best rationale __without__ any few-shot examples, i.e. zero-shot

For zero-shot mode, feel free to use wild prompt-engineering or modify the inference procedure.

### Option 2: Is this prompting tecnique reliable?

_Inspired by research by Anton Voronov, Lena Volf and Max Ryabinin._

For this option, you need to check if the model behavior (and hence, accuracy) is robust to perturbations in the input prompt.

a. Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)

b. Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?



### Option 3: Inference Matters

There are many ways to inference the model, not all of them equal.

a. check whether greedy inference or beam search affects model generation quality

b. implement and evaluate sampling with voting (see explanation below).


The voting technique(b) should work as follows: first, you generate k (e.g. 50) "attempts" at an answer using nucleus sampling (or a similar technique).
Then, you count how many of those attempts chose a particular option (A, B, etc) as the final answer. The option that was chosen most frequently has the most "votes", and therefore "wins".

To speed up voting, you may want to generate these attempts in parallel as a batch. That should be very easy to implement: just run `model.generate` on a list with multiple copies of the same prompt.




================================================

__Common rules:__ You will need to test both hypothes (A and B) in the chosen option. You may choose to replace one of them with your own idea - but please ask course staff in advance (via telegram) if you want full points.

Feel free to organize your code and report as you see fit - but please make sure it's readable and the code runs top-to-bottom :)
Write a short informal report about what you tried and, in doing so, what did you found. Minimum of 2 paragraphs; more is ok; creative visualizations are welcome.

You are allowed (but not required) to prompt the model into generating a report for you --- or helping you write one. However, if you do so, make sure that it is still human-readable :)



In [ ]:
# feel free to organize your solution as you see fit